# CC3104 – Aprendizaje por Refuerzo
## Laboratorio 1 — Task 2 — Preguntas de análisis técnico (Entrega Parcial)

**Integrantes:**
- Sergio Orellana — 221122
- Rodrigo Mansilla — 22611
- Ricardo Chuy — 221007

---


### Instrucciones

Respondan las siguientes preguntas con argumentación técnica, tomando como base el diseño del MDP realizado en el Task 1.

---
### 1. Violaciones de la propiedad de Markov

Identifiquen **al menos dos situaciones** en las que la propiedad de Markov se violaría con su representación de estado actual. Para cada una, propongan una extensión del estado que restaure la propiedad y discutan el costo computacional de esa extensión.

### Situación 1, Otros drones en la flota

La empresa opera una flota completa de drones simultáneamente. Nuestro estado actual **s = (posición, estado_misión, nivel_batería)** no incluye la posición de los otros drones operando al mismo tiempo. Esto viola Markov porque dos drones en la misma posición, con la misma misión y batería, deberían tomar decisiones distintas dependiendo de dónde están los demás por ejemplo, evitar celdas ocupadas por otros drones.

**Extensión propuesta:**

Agregar la posición de cada drone adicional al estado:

- **s = (posición, estado_misión, nivel_batería, posición_drone2, posición_drone3, ...)**

**Costo computacional:**

- El crecimiento es bastante rande. Con solo 2 drones adicionales en una cuadrícula de 25 celdas:
- 1,100 × 25 × 25 = **687,500 estados**

Para flotas grandes este crecimiento hace que el modelo crezca demasiado.

---

### Situación 2, Desgaste acumulado del drone

Nuestro estado incluye el nivel de batería actual, pero no captura el desgaste acumulado durante la operación. Un drone que lleva varias misiones ese día consume batería de forma diferente o bien el drone puede tener cierto desgaste a uno en su primera misión. Dos drones con **s = (2,2, en_camino, bat=5)** idéntico pero con distinto desgaste acumulado se comportarán diferente en la realidad, pero el modelo los trata igual. Eso viola Markov.

**Extensión propuesta:**

Agregar el nivel de desgaste acumulado al estado:

- **s = (posición, estado_misión, nivel_batería, desgaste)**
- Donde desgaste ∈ {0, 1, 2, 3, 4, 5}

**Costo computacional:**

- El crecimiento es manejable: 1,100 × 6 = **6,600 estados**
- Esta extensión es razonable y no compromete la viabilidad del modelo.

---
### 2. Observabilidad completa vs. POMDP

En el dominio de logística urbana, ¿qué tan razonable es el supuesto de que el agente puede observar el estado completo? ¿Qué variables del estado real probablemente no son directamente observables por el drone? ¿Cómo afecta eso la validez del modelo MDP versus un modelo POMDP?

**Respuesta:**

Nuestro modelo MDP asume que el drone conoce su estado s = (posición, estado_misión, nivel_batería) con certeza perfecta en todo momento. Sin embargo, en el mundo real esto es una simplificación bastante grande hay algunas cosas que no son completamente observables y que hemos resumido para poder simplifcar el problema

Variables que probablemente no son directamente observables

- Posición: El mundo real no está dividido en celdas discretas. El drone usa GPS para ubicarse, pero en una ciudad con edificios altos las señales GPS tienen errores e imprecisiones. El drone nunca sabe con certeza exacta en qué celda está puede estar en el límite entre dos celdas y recibir lecturas distintas en cada momento.

- Nivel de batería: Los sensores de batería reportan un porcentaje aproximado, no un valor exacto. Además, como discutimos anteriormente, el desgaste acumulado hace que la batería se comporte de forma impredecible un drone con batería=5 pero muy desgastado puede quedarse sin energía mucho antes que uno nuevo con el mismo nivel.

- Estado del paquete: El drone podría perder el paquete en el camino sin saberlo inmediatamente. Podría llegar al destino creyendo que está "en_camino con paquete" cuando en realidad ya no lo lleva. Nuestro estado_misión no captura esta posibilidad.

**MDP vs POMDP**

Estas limitaciones convierten nuestro problema en realidad en un POMDP (Partially Observable MDP) un MDP donde el agente no puede observar su estado completo con certeza, sino que solo tiene observaciones parciales o ruidosas de él.

La diferencia principal es que en MDP el drone conoce su estado exacto y toma decisiones con certeza mientras que en POMDP el drone solo tiene estimaciones imperfectas de su estado y debe tomar decisiones bajo esa incertidumbre adicional. También nuestro modelo MDP es más simple de implementar y resolver, pero sacrifica fidelidad a la realidad. Un POMDP capturaría mejor el problema real pero sería más complejo de implementar.


---
### 3. ¿Tarea episódica o continua?

Argumenten si este problema debería modelarse como una tarea episódica o continua. ¿Cómo cambia esa decisión el diseño de la función de recompensa y el valor de 𝛾?

**Respuesta:**

_(Argumenten a favor de episódica o continua, y expliquen el impacto en r(s,a,s') y en γ.)_
